In [4]:

from pathlib import Path
import pandas as pd
from getpass import getpass
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

PROJECT_ROOT = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

CLEAN_DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "airline_satisfaction_clean.csv"
)

print("Project folder:", PROJECT_ROOT)
print("Cleaned CSV exists:", CLEAN_DATA_PATH.exists())

if CLEAN_DATA_PATH.exists():
    csv_df = pd.read_csv(CLEAN_DATA_PATH)
    print("CSV dimensions:", csv_df.shape)


Project folder: d:\Airline-Passenger-Satisfaction
Cleaned CSV exists: True
CSV dimensions: (129880, 24)


In [8]:
%pip install pymysql

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:

from getpass import getpass
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

DB_HOST = "localhost"
DB_PORT = 3306
DB_NAME = "airline_analysis"
DB_USER = "root"
DB_PASSWORD = getpass("Enter your MySQL password: ")

connection_url = URL.create(
    drivername="mysql+pymysql",
    username=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=DB_PORT,
    database=DB_NAME
)

engine = create_engine(connection_url)

with engine.connect() as connection:
    result = connection.execute(text("SELECT DATABASE()"))
    print("Connected database:", result.scalar())


Connected database: airline_analysis


In [10]:
%pip install sqlalchemy pymysql

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:

with engine.connect() as connection:
    table_check = connection.execute(
        text("""
            SELECT COUNT(*)
            FROM information_schema.tables
            WHERE table_schema = :database_name
              AND table_name = :table_name
        """),
        {
            "database_name": DB_NAME,
            "table_name": "passengers"
        }
    ).scalar()

    print("Passengers table exists:", table_check == 1)

    if table_check == 1:
        row_count = connection.execute(
            text("SELECT COUNT(*) FROM passengers")
        ).scalar()

        print("Passenger records in MySQL:", row_count)

        sample_df = pd.read_sql(
            text("SELECT * FROM passengers LIMIT 5"),
            connection
        )
        display(sample_df)


Passengers table exists: True
Passenger records in MySQL: 129880


,id,gender,age,customer_type,type_of_travel,class,flight_distance,departure_delay,arrival_delay,departure_and_arrival_time_convenience,...,on_board_service,seat_comfort,leg_room_service,cleanliness,food_and_drink,in_flight_service,in_flight_wifi_service,in_flight_entertainment,baggage_handling,satisfaction
0,1,Male,48,First-time,Business,Business,821,2.0,5.0,3,...,3,5,2,5,5,5,3,5,5,Neutral or Dissatisfied
1,2,Female,35,Returning,Business,Business,821,26.0,39.0,2,...,5,4,5,5,3,5,2,5,5,Satisfied
2,3,Male,41,Returning,Business,Business,853,0.0,0.0,4,...,3,5,3,5,5,3,4,3,3,Satisfied
3,4,Male,50,Returning,Business,Business,1905,0.0,0.0,2,...,5,5,5,4,4,5,2,5,5,Satisfied
4,5,Female,49,Returning,Business,Business,3470,0.0,1.0,3,...,3,4,4,5,4,3,3,3,3,Satisfied


In [12]:

with engine.connect() as connection:
    satisfaction_check = pd.read_sql(
        text("""
            SELECT
                satisfaction,
                COUNT(*) AS passenger_count
            FROM passengers
            GROUP BY satisfaction
            ORDER BY passenger_count DESC
        """),
        connection
    )

display(satisfaction_check)


,satisfaction,passenger_count
0,Neutral or Dissatisfied,73452
1,Satisfied,56428


In [14]:

# # MySQL Import and Validation

# ## Objective
# Load the cleaned airline passenger dataset into MySQL and validate the imported records.

# ## Workflow
# 1. Prepared the cleaned passenger dataset in Python.
# 2. Imported the dataset into MySQL using MySQL Workbench.
# 3. Verified that the passengers table exists.
# 4. Checked the record count and satisfaction categories.
# 5. Used SQL queries to support business analysis.

# ## Database
# Database: airline_analysis
# Table: passengers

# The validation queries retrieve existing data only. They do not recreate or replace database tables.
